# 2 · Set and verify power in standby

Learn the difference between a requested setpoint (`?SP`) and reported power (`?P`), and verify that a command took effect without enabling the laser.

**Self-contained notebook · simulator default · optional human-operated hardware**

From the repository root, activate your virtual environment, install with `python -m pip install -e ".[tutorials]"`, then launch `python -m jupyterlab examples/tutorials`. Select this environment's Python kernel. Run cells top to bottom with Shift+Enter, or choose **Restart Kernel and Run All Cells**. See [setup and troubleshooting](README.md#start-here).

Every demonstration and hardware helper is defined below. The notebook imports only the controller library and Python's standard library; no tutorial script is loaded. Run cells from top to bottom.

[All four tutorials](README.md)

## 1. Command sequence

`?L, ?S, ?F` check STANDBY, closed shutter and faults → `P=0.2500` → `?SP` readback → `?L, ?S, ?P` verification → reject 0.6 W locally → `?SP` → repeat the guarded sequence for `P=0.0000`.

Neither `L=1` nor `S=1` is sent. Commands use four decimal places; readback is compared with the value actually transmitted.

The controller supplies CR/LF framing and waits for each reply. Queries start with `?`; writes use `=`. No `OK` acknowledgment is assumed. Source: supplied [Verdi manual](../../verdi.manual_v5.pdf), Tables 5-1, 5-3 and 5-4; [protocol mapping](../../docs/PROTOCOL.md).

## 2. Import the controller API

Imports do not discover ports, open connections or send commands.

In [ ]:
"""Tutorial 2: set and read back a bounded setpoint while remaining in standby."""

from coherent_verdi import SimulatedVerdi, VerdiController, VerdiError

## 3. Define the application steps

Each cell defines one small function. The complete sequence is run in section 4.

### 3.1 Set and verify a standby setpoint

Check the starting state, send a bounded setpoint, and verify its readback. This function never enables the laser.

In [ ]:
def set_standby_power(laser: VerdiController, target_w: float) -> float:
    if laser.read_laser_state() != 0 or laser.read("?S") != 0:
        raise RuntimeError("Require STANDBY and a closed shutter before changing the setpoint.")
    if laser.read_faults():
        raise RuntimeError("Active faults require diagnosis before changing settings.")
    laser.set_power_w(target_w)  # Validates finite value, ceiling and rounded wire value.
    readback = float(laser.read("?SP"))  # ?SP, not the measured-power query ?P.
    expected = float(f"{target_w:.4f}")
    if readback != expected:
        raise RuntimeError(f"Setpoint mismatch: requested {expected:.4f} W, read {readback} W.")
    if laser.read_laser_state() != 0 or laser.read("?S") != 0:
        raise RuntimeError("State changed unexpectedly; stop and verify with the operator.")
    print(f"Verified setpoint: {readback:.4f} W; STANDBY; shutter closed.")
    print(f"Reported power: {laser.read_power_w():.3f} W (separate from the setpoint).")
    return readback

## 4. Run the simulator

The fixture setup below is synthetic. The exercise target of 0.25 W and ceiling of 0.5 W are not physical safety limits.

### 4.1 Prepare the simulator demonstration

The function below owns the simulator and its controller lifetime. Each call starts a fresh exercise and leaves no worker or open session.

In [ ]:
def simulate_set_power() -> None:
    laser = SimulatedVerdi("V5", allow_writes=True, power_limit_w=0.5)
    try:
        with laser:
            set_standby_power(laser, 0.25)
            try:
                laser.set_power_w(0.6)  # Above this exercise's 0.5 W software ceiling.
            except ValueError as exc:
                print(f"Expected rejection before transmission: {exc}")
            print(f"Setpoint after rejection: {laser.read('?SP'):.4f} W")
            set_standby_power(laser, 0.0)  # Explicit normal completion, not implicit close.
    except (VerdiError, ValueError, RuntimeError) as exc:
        print(f"STOP: {type(exc).__name__}: {exc}. No retry or further state changes.")
        raise
    print("Connection released; no enable or shutter-open command was sent.")

### 4.2 Execute the demonstration

Run this short cell to call the functions just defined. Rerun it to begin with a fresh simulator.

In [ ]:
simulate_set_power()

## 5. Expected simulator result

```text
Verified setpoint: 0.2500 W; STANDBY; shutter closed.
Reported power: 0.000 W (separate from the setpoint).
Expected rejection before transmission: ...
Setpoint after rejection: 0.2500 W
Verified setpoint: 0.0000 W; STANDBY; shutter closed.
Reported power: 0.000 W (separate from the setpoint).
```
The rejected value never reaches the simulator. A zero reading here is simulator policy, not evidence of physical beam safety.

## 6. Try one small change

Change the first target from `0.25` to `0.1` and rerun. Then replace the deliberately rejected `0.6` with `float('nan')` to see nonfinite input rejected by the same `ValueError` handler. Keep the explicit zero-setpoint step.

## 7. Optional real-hardware session for a human operator

Complete the [hardware review procedure](../../HARDWARE_VALIDATION.md) first. Install `python -m pip install -e ".[tutorials,serial]"` in this environment. The hardware functions below are fully visible and call the application functions in section 3 directly.

Fill in the actual model, native port and matching baud. Supply the approved target and ceiling in W; no simulator power default is reused. Confirm cooling, warmup, interlocks, beam conditions and the physical abort procedure. Keep `RUN_HARDWARE=False` for ordinary Run All.

After you enable it, **CONNECT** permits the selected connection and one `?SV` query. Verify the reported device and version. **RUN** permits the displayed lesson. Any other answer cancels. These prompts confirm intent, not site approval.

### 7.1 Import serial interfaces

These imports alone perform no I/O. `contextmanager` lets a `with` block release the connection even on an exception.

In [ ]:
from collections.abc import Iterator
from contextlib import contextmanager

### 7.2 Validate the power configuration

Validation happens before connection. Read-only sessions reject power settings; write sessions require an explicit approved ceiling and target.

In [ ]:
def validate_power(
    model: str, target_w: float | None, power_limit_w: float | None, *, allow_writes: bool
) -> None:
    """Require site values before a physical write; simulator defaults are not limits."""
    if model not in ("V2", "V5", "V6"):
        raise ValueError("model must be V2, V5 or V6")
    if not allow_writes:
        if target_w is not None or power_limit_w is not None:
            raise ValueError("Read-only lessons do not accept power settings")
        return
    if (
        isinstance(power_limit_w, bool)
        or not isinstance(power_limit_w, (int, float))
        or not 0 <= power_limit_w <= float(model[1:])
    ):
        raise ValueError("Supply the approved power_limit_w within the model rating")
    if (
        isinstance(target_w, bool)
        or not isinstance(target_w, (int, float))
        or not 0 <= target_w <= power_limit_w
        or float(f"{target_w:.4f}") > power_limit_w
    ):
        raise ValueError("target_w must be finite, nonnegative and within the approved ceiling")

### 7.3 Connect, identify and release

This helper prompts before opening, sends `?SV` first and closes communication when the `with` block ends. An error stops further commands. **Connection close is not physical shutdown**; use the operator's abort procedure when state is uncertain.

In [ ]:
@contextmanager
def hardware_connection(laser: VerdiController) -> Iterator[VerdiController | None]:
    """Operator confirmation, passive connect, one ?SV, and deterministic disconnect."""
    if input("With candidate/connection approval recorded, type CONNECT: ").strip() != "CONNECT":
        print("Cancelled before opening the port.")
        yield None
        return
    try:
        with laser:
            print(f"Reported software: {laser.read('?SV')}")
            yield laser
    except BaseException:
        print("STOP: state may be unknown. No retries or blind cleanup writes.")
        print("Use the operator's approved physical abort procedure.")
        raise
    finally:
        print("Connection released; disconnect does not shut down the laser.")

### 7.4 Enter the operator's settings

Nothing connects when you run this configuration cell. Set `RUN_HARDWARE=True` only for an attended, approved session. Restore False afterwards.

The manual does not specify the no-active-fault reply for `?F`. Keep `ACTIVE_FAULT_CLEAR_REPLY=None` until Stage 1 records its exact meaning for this firmware. Then enter the verified text here; otherwise a clear-looking response stops the physical session. `SYSTEM OK` is documented for `?FH` only. Simulator defaults use their explicit fixture convention.

In [ ]:
RUN_HARDWARE = False  # Change only for an approved, attended physical session.
HARDWARE_PORT = None  # Set to the operator-confirmed native port, e.g. "COM3".
HARDWARE_MODEL = None  # Set to 'V2', 'V5' or 'V6' after checking the label.
HARDWARE_BAUDRATE = None  # Set to the actual front-panel baud rate.
HARDWARE_TIMEOUT_S = 1.0  # Adjust to the validated transaction deadline.
TARGET_W = None  # Operator-approved requested setpoint, in W.
POWER_LIMIT_W = None  # Operator-approved ceiling, in W.
ACTIVE_FAULT_CLEAR_REPLY = None  # Set only after Stage 1 verifies the exact ?F clear text.

### 7.5 Run the visible hardware sequence

Calls `set_standby_power` from section 3. The supplied setpoint remains stored in standby; no rejected-input or return-to-zero exercise is run.

The call completes in one cell; no connection waits between cells. A new run prompts again.

In [ ]:
if RUN_HARDWARE:
    model = HARDWARE_MODEL
    validate_power(model, TARGET_W, POWER_LIMIT_W, allow_writes=True)
    controller = VerdiController(
        HARDWARE_PORT,
        model=model,
        baudrate=HARDWARE_BAUDRATE,
        timeout_s=HARDWARE_TIMEOUT_S,
        active_fault_clear_reply=ACTIVE_FAULT_CLEAR_REPLY,
        allow_writes=True,
        power_limit_w=POWER_LIMIT_W,
    )
    print(f"HARDWARE: {model}, port={HARDWARE_PORT}, baud={HARDWARE_BAUDRATE}")
    print("Plan: set the approved target in STANDBY and verify; retain the target.")
    print(f"Target: {TARGET_W} W; ceiling: {POWER_LIMIT_W} W.")
    with hardware_connection(controller) as laser:
        if laser is not None:
            if input("Verify device/version and approved scope; type RUN: ").strip() == "RUN":
                set_standby_power(laser, TARGET_W)
            else:
                print("Cancelled after ?SV; no lesson commands sent.")
else:
    print("Hardware section skipped. Set explicit operator configuration to use it.")

## 8. What this establishes

Simulator runs verify software behavior, not physical response or calibration. A status sample is sequential, not an interlock or proof of a safe beam path. Review the [simulator assumptions](../../docs/SIMULATOR.md) and [operator guide](README.md#human-operated-hardware) before physical use.